In [ ]:
BIBLIA_INPUT_PDF = "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.pdf"
BIBLIA_TXT_EXTRACTED = "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"
BIBLIA_OUTPUT = "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"

In [ ]:
PROCESSAR_PDF = True
PROCESSAR_TXT = True

In [ ]:
import pdfplumber

def extrair_texto_pdf(pdf_path, txt_path):
    with pdfplumber.open(pdf_path) as pdf:
        with open(txt_path, "w", encoding="utf-8") as f_out:
            for i, page in enumerate(pdf.pages):
                if i < 2:
                    continue
                elif i == 2904:
                    break
                texto = page.extract_text()
                if texto:
                    f_out.write(texto + "\n")
    print(f"✅ Texto extraído para: {txt_path}")

if PROCESSAR_PDF:

    extrair_texto_pdf(BIBLIA_INPUT_PDF, BIBLIA_TXT_EXTRACTED)

In [ ]:
def remover_3_primeiras_linhas(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f_in:
        linhas = f_in.readlines()
    with open(txt_path, "w", encoding="utf-8") as f_out:
        f_out.writelines(linhas[3:])
    print(f"✅ Texto processado para: {txt_path}")
    
if PROCESSAR_PDF and PROCESSAR_TXT:
    remover_3_primeiras_linhas(BIBLIA_TXT_EXTRACTED)

In [ ]:
def remover_trechos_indesejados(caminho_arquivo):
    trecho_alvo_1 = "Bíblia Ave-Maria"
    trecho_alvo_2 = "© 1959 Editora Ave-Maria."
    remover_linhas = False

    with open(caminho_arquivo, "r", encoding="utf-8") as f:
        linhas = f.readlines()

    linhas_filtradas = []
    for linha in linhas:
        if trecho_alvo_1 in linha:
            remover_linhas = True
        elif remover_linhas and trecho_alvo_2 in linha:
            continue
        elif remover_linhas and linha.strip().isdigit():
            remover_linhas = False
            continue
        if not remover_linhas:
            linhas_filtradas.append(linha)

    with open(caminho_arquivo, "w", encoding="utf-8") as f:
        f.writelines(linhas_filtradas)

    print(f"✅ Trechos indesejados removidos de {caminho_arquivo}")


if PROCESSAR_TXT:
    remover_trechos_indesejados(BIBLIA_TXT_EXTRACTED)

In [ ]:
from copy import copy


def transformar_biblia_versiculo_a_versiculo(caminho_entrada, caminho_saida):
    with open(caminho_entrada, "r", encoding="utf-8") as f:
        linhas = f.readlines()

    livro_atual = ""
    capitulo_atual = ""
    versiculo_numero = 0
    versiculos = []
    i = 0

    versiculo = ""
    while i < len(linhas):
        linha = linhas[i].strip()
        
        # Detecta novo livro: linha não vazia, sem dois pontos, e próxima linha é "Livro Capítulo"
        if linha and i + 1 < len(linhas) and linhas[i + 1].strip().startswith(linha):
            if livro_atual and capitulo_atual and versiculo:    
                versiculos.append(
                    f"{livro_atual} {capitulo_atual}:{versiculo_numero} {versiculo.strip()}"
                )
            livro_atual = copy(linha)
            # print(f"📖 Livro atual: {livro_atual}")
            i += 1
            versiculo = ""
            continue

        # Detecta novo capítulo (ex: Gênesis 1)
        if (
            livro_atual
            and linha.startswith(livro_atual)
            and linha[len(livro_atual) :].strip().isdigit()
        ):
            capitulo_atual = linha[len(livro_atual) :].strip()
            if livro_atual and capitulo_atual and versiculo:
                versiculos.append(
                    f"{livro_atual} {capitulo_atual}:{versiculo_numero} {versiculo.strip()}"
                )
            # print(f"📖 Capítulo atual: {capitulo_atual}")
            i += 1  # Avança para a próxima linha
            versiculo = ""
            continue
        
        # Detecta número do versículo
        if linha.strip().isdigit():
            if livro_atual and capitulo_atual and versiculo:    
                versiculos.append(
                    f"{livro_atual} {capitulo_atual}:{versiculo_numero} {versiculo.strip()}"
                )
            i += 1
            versiculo_numero = linha.strip()
            # print(f"📖 Livro: {livro_atual}\tCapítulo: {capitulo_atual}\tVersículo: {versiculo_numero}")
            versiculo = ""
            continue
            
        versiculo += linha + " "
        i += 1
    
    if livro_atual and capitulo_atual and versiculo:    
        versiculos.append(
            f"{livro_atual} {capitulo_atual}:{versiculo_numero} {versiculo.strip()}"
        )
    
    print(versiculos[11000:11001])

    with open(caminho_saida, "w", encoding="utf-8") as f:
        for versiculo in versiculos:
            f.write(versiculo + "\n")

    print(
        f"✅ Transformação concluída. {len(versiculos)} versículos salvos em {caminho_saida}"
    )


if PROCESSAR_TXT:
    transformar_biblia_versiculo_a_versiculo(
        BIBLIA_TXT_EXTRACTED,
        BIBLIA_OUTPUT,
    )